In [1]:
# Clone the repository and navigate into it, if the repo is private you will need to set up authentication (e.g., using SSH keys or personal access tokens).
!git clone https://github.com/Camel-glitch/project.git
%cd project

Cloning into 'project'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 57 (delta 33), reused 39 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 13.03 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/project


In [13]:
# Update the repository to get the latest changes
!git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 302 bytes | 302.00 KiB/s, done.
From https://github.com/Camel-glitch/project
   a91ec73..a0a1cea  main       -> origin/main
Updating a91ec73..a0a1cea
Fast-forward
 MC_euler.h | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [9]:
# Check the status of the GPU
!nvidia-smi

Fri Mar 20 11:08:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd
import numpy as np

# Plages de paramètres
kappas = np.linspace(0.1, 10, 15)
thetas = np.linspace(0.01, 0.5, 10)
sigmas = np.linspace(0.1, 1.0, 10)

valid_points = []

for k in kappas:
    for t in thetas:
        for s in sigmas:
            # Condition de Feller : 2*kappa*theta > sigma^2
            if 2 * k * t > s**2:
                valid_points.append({'kappa': k, 'theta': t, 'sigma': s})

df = pd.DataFrame(valid_points)
df.to_csv('params_feller.csv', index=False, header=False)
print(f"{len(df)} points valides générés dans params_feller.csv")

In [ ]:
!make clean

In [10]:
!make

nvcc -c gamma.cu -o gamma.o
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
At end of source: error: expected a "}"
gamma.cu(8): note #3196-D: to match this "{"
  __attribute__((device)) double generate_gamma(double a, curandState *state) {
                                                                              ^

1 error detected in the compilation of "gamma.cu".
make: *** [Makefile:26: gamma.o] Error 2


In [ ]:
#Stop it with ctrl+C if you don't want all the fellers parameters from the csv file 
!./main > results.csv 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Chargement des données
df = pd.read_csv('results.csv')

# 2. Calcul des erreurs par rapport à l'Exact
methods = ['e', 'a', 'a30']
names = {'e': 'Euler (N=100)', 'a': 'Almost Fine (N=100)', 'a30': 'Almost Coarse (N=30)'}

for m in methods:
    df[f'bias_{m}'] = np.abs(df[f'p_{m}'] - df['p_ex'])
    df[f'rel_err_{m}'] = (df[f'bias_{m}'] / df['p_ex']) * 100

# --- VISUALISATION 1 : Erreur vs Temps (Le Front de Pareto) ---
plt.figure(figsize=(10, 6))
for m in methods:
    plt.scatter(df[f'ms_{m}'], df[f'bias_{m}'], label=names[m], alpha=0.6)

plt.yscale('log')
plt.xscale('log')
plt.xlabel('Temps d\'exécution (ms) - Log')
plt.ylabel('Biais absolu vs Exact - Log')
plt.title('Analyse du compromis Précision / Vitesse')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

# --- VISUALISATION 2 : Impact de la Condition de Feller ---
# On regarde si l'erreur d'Euler explose quand la marge de Feller est faible
plt.figure(figsize=(10, 6))
sns.lineplot(data=df, x='feller_margin', y='bias_e', label='Biais Euler')
sns.lineplot(data=df, x='feller_margin', y='bias_a30', label='Biais Almost Coarse')
plt.title('Sensibilité du biais à la condition de Feller')
plt.xlabel('Marge de Feller (2κθ - σ²)')
plt.ylabel('Biais')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Chargement et nettoyage
df = pd.read_csv('results.csv')

# 2. Calcul des métriques
methods = {'e': 'Euler', 'a': 'Almost (Fine)', 'a30': 'Almost (Coarse)'}

for m in methods.keys():
    # Biais relatif en % par rapport à l'Exact
    df[f'rel_bias_{m}'] = np.abs((df[f'p_{m}'] - df['p_ex']) / df['p_ex']) * 100
    # On renomme les colonnes de temps pour la légende
    df[f'time_{m}'] = df[f'ms_{m}']

# 3. Préparation des données pour le Boxplot (Format Long)
# On crée un DataFrame spécifique pour le Biais
df_bias = df.melt(value_vars=[f'rel_bias_{m}' for m in methods.keys()],
                  var_name='Méthode', value_name='Biais Relatif (%)')
df_bias['Méthode'] = df_bias['Méthode'].map(lambda x: methods[x.replace('rel_bias_', '')])

# On crée un DataFrame spécifique pour le Temps
df_time = df.melt(value_vars=[f'time_{m}' for m in methods.keys()] + ['ms_ex'],
                  var_name='Méthode', value_name='Running Time (ms)')
# Mapping des noms pour le temps (incluant l'Exact)
time_names = {**{f'time_{k}': v for k, v in methods.items()}, 'ms_ex': 'Exact'}
df_time['Méthode'] = df_time['Méthode'].map(time_names)

# 4. Affichage des Graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Graphique 1 : Biais Relatif
sns.boxplot(data=df_bias, x='Méthode', y='Biais Relatif (%)', ax=ax1, palette='Set2')
ax1.set_title('Distribution du Biais Relatif (%)')
#ax1.set_yscale('log') # Log scale souvent utile si Euler a de gros outliers
ax1.grid(True, axis='y', ls='--', alpha=0.7)

# Graphique 2 : Running Time
sns.boxplot(data=df_time, x='Méthode', y='Running Time (ms)', ax=ax2, palette='Set3')
ax2.set_title('Distribution du Temps d\'Exécution (ms)')
ax2.grid(True, axis='y', ls='--', alpha=0.7)

plt.tight_layout()
plt.show()

Why writing device function ?
nvcc can replace the code function when thread is calling this function (inline)
which make it faster because there is no data transfer.

!nvcc-smi